<a href="https://colab.research.google.com/github/silvanhess/manifesto_analysis/blob/main/BERTopic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tutorial** - Topic Modeling with BERTopic
(last updated 01-09-2022)

In this tutorial we will be exploring how to use BERTopic to create topics from the well-known 20Newsgroups dataset. The most frequent use-cases and methods are discussed together with important parameters to keep a look out for.


## BERTopic
BERTopic is a topic modeling technique that leverages 🤗 transformers and a custom class-based TF-IDF to create dense clusters allowing for easily interpretable topics whilst keeping important words in the topic descriptions.

<br>

<img src="https://raw.githubusercontent.com/MaartenGr/BERTopic/master/images/logo.png" width="40%">

# Enabling the GPU

First, you'll need to enable GPUs for the notebook:

- Navigate to Edit→Notebook Settings
- select GPU from the Hardware Accelerator drop-down

[Reference](https://colab.research.google.com/notebooks/gpu.ipynb)

# **Installing BERTopic**

We start by installing BERTopic from PyPi:

In [ ]:
%%capture
!pip install bertopic

## Restart the Notebook
After installing BERTopic, some packages that were already loaded were updated and in order to correctly use them, we should now restart the notebook.

From the Menu:

Runtime → Restart Runtime

# Data
For this example, we use the popular 20 Newsgroups dataset which contains roughly 18000 newsgroups posts

In [ ]:
from sklearn.datasets import fetch_20newsgroups
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

# **Topic Modeling**

In this example, we will go through the main components of BERTopic and the steps necessary to create a strong topic model.




## Training

We start by instantiating BERTopic. We set language to `english` since our documents are in the English language. If you would like to use a multi-lingual model, please use `language="multilingual"` instead.

We will also calculate the topic probabilities. However, this can slow down BERTopic significantly at large amounts of data (>100_000 documents). It is advised to turn this off if you want to speed up the model.


In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(docs)

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
2026-01-13 14:27:26,828 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/589 [00:00<?, ?it/s]

2026-01-13 14:28:15,313 - BERTopic - Embedding - Completed ✓
2026-01-13 14:28:15,314 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-13 14:28:50,101 - BERTopic - Dimensionality - Completed ✓
2026-01-13 14:28:50,103 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-13 14:29:42,413 - BERTopic - Cluster - Completed ✓
2026-01-13 14:29:42,429 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-13 14:29:46,189 - BERTopic - Representation - Completed ✓


**NOTE**: Use `language="multilingual"` to select a model that support 50+ languages.

## Extracting Topics
After fitting our model, we can start by looking at the results. Typically, we look at the most frequent topics first as they best represent the collection of documents.

In [ ]:
freq = topic_model.get_topic_info(); freq.head(5)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6631,-1_to_the_of_you,"[to, the, of, you, and, is, in, it, for, that]","[Enclosed are the rules, guidelines and relate..."
1,0,1828,0_game_team_games_he,"[game, team, games, he, players, season, hocke...","[Apparently, Part 2 (defensemen numbered 2 thr..."
2,1,623,1_key_clipper_chip_encryption,"[key, clipper, chip, encryption, keys, escrow,...",[\n > Clipper might be a good way to cover th...
3,2,476,2_israel_israeli_jews_arab,"[israel, israeli, jews, arab, jewish, arabs, p...",[From: Center for Policy Research <cpr>\nSubje...
4,3,433,3_card_monitor_video_vga,"[card, monitor, video, vga, drivers, diamond, ...",[I have a Radius Precision Color 24x video car...


-1 refers to all outliers and should typically be ignored. Next, let's take a look at a frequent topic that were generated:

In [ ]:
topic_model.get_topic(0)  # Select the most frequent topic

[('game', np.float64(0.010327350332050242)),
 ('team', np.float64(0.008991079215501421)),
 ('games', np.float64(0.0071641478370298535)),
 ('he', np.float64(0.007023307285265203)),
 ('players', np.float64(0.006297725125833217)),
 ('season', np.float64(0.006204423384166541)),
 ('hockey', np.float64(0.006105894363568919)),
 ('play', np.float64(0.005762222998092491)),
 ('25', np.float64(0.005625034661566436)),
 ('year', np.float64(0.005577960481606828))]

**NOTE**: BERTopic is stocastich which mmeans that the topics might differ across runs. This is mostly due to the stocastisch nature of UMAP.

In [ ]:
### Attributes

## Attributes

There are a number of attributes that you can access after having trained your BERTopic model:


| Attribute | Description |
|------------------------|---------------------------------------------------------------------------------------------|
| topics_               | The topics that are generated for each document after training or updating the topic model. |
| probabilities_ | The probabilities that are generated for each document if HDBSCAN is used. |
| topic_sizes_           | The size of each topic                                                                      |
| topic_mapper_          | A class for tracking topics and their mappings anytime they are merged/reduced.             |
| topic_representations_ | The top *n* terms per topic and their respective c-TF-IDF values.                             |
| c_tf_idf_              | The topic-term matrix as calculated through c-TF-IDF.                                       |
| topic_labels_          | The default labels for each topic.                                                          |
| custom_labels_         | Custom labels for each topic as generated through `.set_topic_labels`.                                                               |
| topic_embeddings_      | The embeddings for each topic if `embedding_model` was used.                                                              |
| representative_docs_   | The representative documents for each topic if HDBSCAN is used.                                                |

For example, to access the predicted topics for the first 10 documents, we simply run the following:

In [ ]:
topic_model.topics_[:10]

[0, 3, -1, 4, 121, -1, -1, 0, 0, -1]

# **Visualization**
There are several visualization options available in BERTopic, namely the visualization of topics, probabilities and topics over time. Topic modeling is, to a certain extent, quite subjective. Visualizations help understand the topics that were created.

## Visualize Topics
After having trained our `BERTopic` model, we can iteratively go through perhaps a hundred topic to get a good
understanding of the topics that were extract. However, that takes quite some time and lacks a global representation.
Instead, we can visualize the topics that were generated in a way very similar to
[LDAvis](https://github.com/cpsievert/LDAvis):

In [ ]:
topic_model.visualize_topics()

## Visualize Topic Probabilities

The variable `probabilities` that is returned from `transform()` or `fit_transform()` can
be used to understand how confident BERTopic is that certain topics can be found in a document.

To visualize the distributions, we simply call:

In [ ]:
topic_model.visualize_distribution(probs[200], min_probability=0.015)

## Visualize Topic Hierarchy

The topics that were created can be hierarchically reduced. In order to understand the potential hierarchical structure of the topics, we can use scipy.cluster.hierarchy to create clusters and visualize how they relate to one another. This might help selecting an appropriate nr_topics when reducing the number of topics that you have created.

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

## Visualize Terms

We can visualize the selected terms for a few topics by creating bar charts out of the c-TF-IDF scores for each topic representation. Insights can be gained from the relative c-TF-IDF scores between and within topics. Moreover, you can easily compare topic representations to each other.

In [ ]:
topic_model.visualize_barchart(top_n_topics=5)

## Visualize Topic Similarity
Having generated topic embeddings, through both c-TF-IDF and embeddings, we can create a similarity matrix by simply applying cosine similarities through those topic embeddings. The result will be a matrix indicating how similar certain topics are to each other.

In [ ]:
topic_model.visualize_heatmap(n_clusters=20, width=1000, height=1000)

## Visualize Term Score Decline
Topics are represented by a number of words starting with the best representative word. Each word is represented by a c-TF-IDF score. The higher the score, the more representative a word to the topic is. Since the topic words are sorted by their c-TF-IDF score, the scores slowly decline with each word that is added. At some point adding words to the topic representation only marginally increases the total c-TF-IDF score and would not be beneficial for its representation.

To visualize this effect, we can plot the c-TF-IDF scores for each topic by the term rank of each word. In other words, the position of the words (term rank), where the words with the highest c-TF-IDF score will have a rank of 1, will be put on the x-axis. Whereas the y-axis will be populated by the c-TF-IDF scores. The result is a visualization that shows you the decline of c-TF-IDF score when adding words to the topic representation. It allows you, using the elbow method, the select the best number of words in a topic.


In [ ]:
topic_model.visualize_term_rank()

# **Topic Representation**
After having created the topic model, you might not be satisfied with some of the parameters you have chosen. Fortunately, BERTopic allows you to update the topics after they have been created.

This allows for fine-tuning the model to your specifications and wishes.

## Update Topics
When you have trained a model and viewed the topics and the words that represent them,
you might not be satisfied with the representation. Perhaps you forgot to remove
stopwords or you want to try out a different `n_gram_range`. We can use the function `update_topics` to update
the topic representation with new parameters for `c-TF-IDF`:


In [ ]:
topic_model.update_topics(docs, n_gram_range=(1, 2))

In [ ]:
topic_model.get_topic(0)   # We select topic that we viewed before

[('game', np.float64(0.006558961749706103)),
 ('team', np.float64(0.005602932057520269)),
 ('he', np.float64(0.005275707278134904)),
 ('games', np.float64(0.0044198115243647)),
 ('the', np.float64(0.003955251529439362)),
 ('players', np.float64(0.003790695107781533)),
 ('was', np.float64(0.0037647195199079493)),
 ('season', np.float64(0.00373300261225644)),
 ('hockey', np.float64(0.0036635659622287387)),
 ('year', np.float64(0.0036031981564795274))]

## Topic Reduction
We can also reduce the number of topics after having trained a BERTopic model. The advantage of doing so,
is that you can decide the number of topics after knowing how many are actually created. It is difficult to
predict before training your model how many topics that are in your documents and how many will be extracted.
Instead, we can decide afterwards how many topics seems realistic:





In [ ]:
topic_model.reduce_topics(docs, nr_topics=60)

2026-01-13 14:30:07,811 - BERTopic - Topic reduction - Reducing number of topics
2026-01-13 14:30:07,877 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-13 14:30:20,255 - BERTopic - Representation - Completed ✓
2026-01-13 14:30:20,295 - BERTopic - Topic reduction - Reduced number of topics from 221 to 60


In [ ]:
# Access the newly updated topics with:
print(topic_model.topics_)

[0, 8, -1, 3, 3, -1, -1, 0, 0, -1, -1, -1, -1, 42, 9, 2, 3, 6, 6, 13, -1, 12, 10, 13, 0, 9, -1, -1, 8, -1, -1, 6, 3, 0, 14, 17, 11, 25, -1, -1, 9, 20, 31, -1, 0, -1, -1, 1, 5, 1, -1, 1, 6, 5, 25, 40, -1, 11, -1, 29, 0, -1, -1, 28, -1, 37, 0, -1, -1, -1, 11, 2, 1, 16, -1, -1, 0, 3, -1, 0, 23, -1, -1, 31, 11, 4, -1, 2, -1, 45, 0, 7, 4, 8, 2, 11, -1, -1, -1, -1, 54, -1, -1, 2, 7, 0, -1, -1, -1, 3, 17, -1, 9, 12, 2, 2, -1, 37, 0, 1, -1, 38, -1, 1, 6, -1, -1, 7, -1, -1, -1, 4, -1, 0, 3, 7, 3, 5, 3, -1, -1, 17, 6, 12, 25, 5, 17, 10, 0, 1, 13, 13, 3, -1, 46, -1, -1, 2, 9, 8, 10, 7, 21, -1, -1, -1, 4, 9, 37, 14, 11, -1, 1, -1, 16, -1, 0, -1, -1, 0, 29, 0, 44, -1, -1, 7, -1, 6, -1, -1, 1, 7, 7, -1, 18, 4, 2, -1, -1, 10, 4, -1, -1, -1, -1, -1, 16, -1, 3, 4, -1, 12, -1, 1, 13, 6, -1, -1, -1, 7, 0, 15, 3, 1, 5, -1, 5, 13, -1, 0, 13, -1, -1, 16, 0, -1, 4, 0, 0, 12, 34, 34, 14, 9, -1, 1, 7, -1, 4, 2, 3, 21, 16, 8, 20, -1, 21, -1, 3, 8, 34, -1, 35, 53, 8, -1, 8, -1, -1, 8, 35, -1, 7, 10, 48, -1, 2, -

# **Search Topics**
After having trained our model, we can use `find_topics` to search for topics that are similar
to an input search_term. Here, we are going to be searching for topics that closely relate the
search term "vehicle". Then, we extract the most similar topic and check the results:

In [ ]:
similar_topics, similarity = topic_model.find_topics("vehicle", top_n=5); similar_topics

[13, 29, 7, 1, 23]

In [ ]:
topic_model.get_topic(71)

False

# **Model serialization**
The model and its internal settings can easily be saved. Note that the documents and embeddings will not be saved. However, UMAP and HDBSCAN will be saved.

In [ ]:
# Save model
topic_model.save("my_model")

2026-01-13 14:30:24,070 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [ ]:
# Load model
my_model = BERTopic.load("my_model")

# **Embedding Models**
The parameter `embedding_model` takes in a string pointing to a sentence-transformers model, a SentenceTransformer, or a Flair DocumentEmbedding model.

## Sentence-Transformers
You can select any model from sentence-transformers here and pass it through BERTopic with embedding_model:



In [ ]:
topic_model = BERTopic(embedding_model="xlm-r-bert-base-nli-stsb-mean-tokens")

Or select a SentenceTransformer model with your own parameters:


In [ ]:
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("distilbert-base-nli-mean-tokens", device="cpu")
topic_model = BERTopic(embedding_model=sentence_model, verbose=True)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Click [here](https://www.sbert.net/docs/pretrained_models.html) for a list of supported sentence transformers models.  
